# 第1部分：项目介绍

# KuzH888-ShoppingAgent

## 项目简介

这是一个基于 HelloAgents 的多语言智能购物客服毕业设计。系统从本地商品目录提取真实商品事实，根据预算、场景、必要功能和偏好进行推荐，并支持商品详情、2–3 件商品比较以及商城政策问答。项目采用 FastAPI 后端与 Vue 3 + TypeScript 前端分离架构。

## 作者信息

- 姓名：
- GitHub：[@KuzH888](https://github.com/KuzH888)
- 日期：2026-09-09

# 第2部分：环境配置

首次配置环境时运行安装单元格。真实 API Key 只允许保存在 `.env`，不要写入 Notebook 或提交至 GitHub。当前默认使用模拟模式，因此本 Notebook 不会产生 API 费用。

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from collections import Counter

from hello_agents import HelloAgentsLLM, SimpleAgent
from hello_agents.tools import Tool, ToolParameter, ToolResponse

from src.utils.config import list_public_models, load_runtime_config

runtime_config = load_runtime_config()
print("运行配置：", runtime_config.public_dict())
print("可选模型：", [model["id"] for model in list_public_models()])

# 第3部分：工具定义

项目使用四个 HelloAgents 工具：商品搜索、商品详情、商品比较和商城政策查询。价格、库存、评分与规格均从通过 Pydantic 验证的本地 JSON 数据读取。

In [ ]:
from src.tools import (
    CompareProductsTool,
    ProductDetailsTool,
    SearchProductsTool,
    StorePolicyTool,
)
from src.utils.catalog import load_catalog

catalog = load_catalog()
tools = [SearchProductsTool(), ProductDetailsTool(), CompareProductsTool(), StorePolicyTool()]
counts = Counter(product.category.value for product in catalog.products)
print(f"商城：{catalog.store_name}；商品总数：{len(catalog.products)}")
print("分类统计：", dict(counts))
print("已创建工具：", [tool.name for tool in tools])

# 第4部分：智能体构建

模拟模式使用确定性的 `ShoppingAssistant`，适合开发、回归测试和演示。最终联网测试时，`create_live_agent()` 将创建 HelloAgents `SimpleAgent` 并注册四个工具。

In [ ]:
from src.agents import ShoppingAssistant, create_live_agent

shopping_assistant = ShoppingAssistant()
if runtime_config.simulation_mode:
    agent = None
    print("模拟模式已启用：不会连接外部 API。")
else:
    agent = create_live_agent(selected_model=runtime_config.model_id)
    print(f"在线智能体已创建：{runtime_config.model_id}")

# 第5部分：功能演示

以下单元格演示推荐、商品详情、比较和政策问答。重新运行时使用不同的会话编号，避免历史需求影响当前示例。

In [ ]:
examples = [
    ("中文推荐", "我需要100澳元以内、适合通勤的耳机，降噪很重要，而且希望轻便。"),
    ("English recommendation", "I need a travel pillow with neck support for a flight, under AUD 50."),
    ("商品详情", "请介绍 DIG-001"),
    ("商品比较", "比较 DIG-001 和 DIG-003"),
    ("商城政策", "请告诉我退换货政策"),
]

for index, (title, query) in enumerate(examples, start=1):
    print(f"\n=== {title} ===")
    if agent is None:
        reply = shopping_assistant.chat(query, session_id=f"notebook-demo-{index}")
        print(reply.message)
    else:
        print(agent.run(query))

# 第6部分：性能评估

本项目将评估拆成两阶段：

1. **开发基线**：完全离线，检查预期推荐、硬性约束、商品事实忠实度、语言一致性和本地响应时间。
2. **最终评估**：替换正式商品和配图后重新生成，并增加真实 LLM 工具调用与网络延迟测试。

下面只计算开发基线，不应直接作为最终毕业设计成绩。

In [ ]:
from src.evaluation import evaluate_project

evaluation = evaluate_project(label="development-baseline")
print("目录 SHA-256：", evaluation["catalogue_sha256"])
print("案例数量：", evaluation["case_count"])
evaluation["metrics"]

# 第7部分：总结与展望

## 项目总结

### 已实现功能

- 24 件双语模拟商品及严格数据验证
- 中英文需求解析、硬性过滤和可解释排序
- 商品搜索、详情、比较和商城政策四个工具
- 会话澄清与中英文结构化回复
- FastAPI 后端和 Vue 3 + TypeScript 商城前端
- 商品详情抽屉、2–3 件商品对比和浮动客服窗口
- 19 个离线评估案例及目录版本指纹

### 已解决挑战

- 使用完整单词边界避免英文短词误匹配
- 无精确匹配时明确标记近似备选，不暗中放宽硬性条件
- 商品事实由 Python 工具提供，减少 LLM 编造价格和库存的风险

### 后续工作

- 手动替换商品数据和商品配图，并同步更新评估案例
- 配置本地 API Key，验证不同 OpenAI 模型的真实工具调用
- 生成最终评估报告、项目截图和毕业设计总结